# Running KBMOD with Noise-Added Images

We seek to gague KBMOD performance on images where there is more noise than usual for our brown dwarfs. We will do this by generating noisy images, running the KBMOD pipeline on said images, and obtaining the final returned likelihood value for our true brown dwarf trajectory.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd

from kbmod.image_collection import ImageCollection
from kbmod.standardizers.fits_standardizers.kbmodv05 import KBMODV0_5, KBMODV0_5Config
from kbmod.configuration import SearchConfiguration
from kbmod.run_search import SearchRunner

In [ ]:
ps1_bit_flag_map = {
    "DETECTOR": 2**0,
    "FLAT": 2**1,
    "DARK": 2**2,
    "BLANK": 2**3,
    "CTE": 2**4,
    "SAT": 2**5,
    "LOW": 2**6,
    "SUSPECT": 2**7,
    "BURNTOOL": 2**8,
    "CR": 2**9,
    "SPIKE": 2**10,
    "GHOST": 2**11,
    "STREAK": 2**12,
    "STARCORE": 2**13,
    "CONV.BAD": 2**14,
    "CONV.POOR": 2**15,
    "MARK": 2**16
}

ps1_mask_flags = ["DETECTOR", "BLANK", "CR", 
                  "SPIKE", "GHOST", "STARCORE",
                  "CONV.BAD", "STREAK", "BURNTOOL"]

load_config = KBMODV0_5Config(mask_flags=ps1_mask_flags, bit_flag_map=ps1_bit_flag_map)

# Loading from the files takes a while (multiple minutes).
ic = ImageCollection.fromDir(filepath, force=KBMODV0_5, config=load_config)
wu = ic.toWorkUnit()
wu.print_stats()

In [ ]:
def kbmod_pipeline(bit_flag_map, mask_flags, filepath, input_parameters):
    load_config = KBMODV0_5Config(mask_flags=mask_flags, bit_flag_map=bit_flag_map)
    ic = ImageCollection.fromDir(filepath, force=KBMODV0_5, config=load_config)
    wu = ic.toWorkUnit()

    config = SearchConfiguration.from_dict(input_parameters)
    wu.config = config
    rs = SearchRunner()
    results = rs.run_search_from_work_unit(wu)

    # TODO: Read results and return likelihood
    likelihood = results.read("likelihood")

    return likelihood

def noise_runs(imgs_path, noise_min, noise_max, noise_steps,
               bit_flag_map, mask_flags, input_parameters):
    assert((noise_min < noise_max) and (noise_steps > 0))

    rows = []
    curr_noise = noise_min

    while (curr_noise < noise_max):
        noise_imgs_path = add_white_noise(imgs_path, sigma=curr_noise)
        likelihood = kbmod_pipeline(bit_flag_map, mask_flags, 
                                    noise_imgs_path, input_parameters)
        rows.append({"sigma": curr_noise, "likelihood": likelihood})
        # Delete generated images

        curr_noise += (noise_max-noise_min)/noise_steps

    df = pd.DataFrame(rows)
    return df
